# Vision Transformers (ViT)

### Contextualização

Até meados de 2020, o estado da arte em visão computacional era dominado quase exclusivamente pelas Redes Neurais Convolucionais (CNNs). Arquiteturas como VGG e ResNet constituíam o padrão para tarefas de classificação de imagens. Essas redes baseiam-se na premissa de que a compreensão de uma imagem pode ser alcançada por meio da análise de regiões locais de pixels vizinhos, permitindo a extração progressiva de características visuais como bordas, texturas e padrões. Esse processo ocorre por meio das operações de convolução, que geram mapas de características capazes de representar diferentes níveis de abstração da imagem [<a href="#ref1">1</a>].

Entretanto, com o sucesso da arquitetura Transformer no processamento de linguagem natural (exemplificada por modelos como BERT e GPT) a forma de analisar imagens, tradicionalmente baseada em arquiteturas de CNNs, passou a seguir um caminho diferente. Esse avanço resultou em modelos mais versáteis, capazes de integrar informações visuais e textuais em um mesmo pipeline multimodal. Nesse contexto, surgiram os Vision Transformers (ViT), que aplicam os princípios dos Transformers ao processamento de imagens, permitindo a extração de características visuais de maneira semelhante ao tratamento dado em técnicas de Processamento de Linguagem Natural (NLP) [<a href="#ref2">2</a>].

### Arquitetura dos Vision Transformers (ViT)

Apesar do sucesso das CNNs, elas apresentam limitações quando o problema exige modelar relações globais na imagem, como dependências espaciais de longo alcance. Em cenários onde o contexto global é tão importante quanto os detalhes locais (e.g, reconhecimento de cenas complexas ou estruturas espalhadas na imagem), essas limitações tornam-se mais evidentes [<a href="#ref2">2</a>].

<figure style="text-align: center;">
  <img src="assets/figura-1.jpg" alt="Figura 1" width="800">
</figure>

**Figura 1** – Arquitetura de Vision Transformers (ViT).

Os Vision Transformers (ViT) surgem como uma adaptação direta dos Transformers ao domínio visual, pois sua origem foi proposta em análises de modelos voltados de NLP. Em vez de processar a imagem como um todo contínuo, o ViT divide a imagem em pequenos *patches* (blocos), como ilustrado na Figura 1. Cada patch é então achatado e projetado em um vetor de dimensão fixa, formando uma sequência de tokens, processo esse análogo às palavras em um texto.

Essa sequência de *patches* é enriquecida com embeddings posicionais, permitindo que o modelo saiba onde cada patch se encontra na imagem. Além disso, é inserido um token especial de classificação ($X_{class}$), cuja representação final será usada para tomar a decisão de classe. O resultado é uma matriz de dimensão:  

$$(𝑁+1) \times 𝑑_{𝑚𝑜𝑑𝑒𝑙}$$ 

Na qual N é o número de *patches*.

Uma vez formada essa sequência, ela é processada por um *Transformer Encoder*, composto por camadas de *self-attention* e *Multilayer Perceptron* (MLP). Diferentemente das CNNs, o mecanismo de *self-attention* permite que cada patch se relacione diretamente com todos os outros *patches*, capturando dependências globais desde as primeiras camadas do modelo.

Ao final do codificador, a representação associada ao token ($X_{class}$) agrega informações relevantes de toda a imagem. Essa representação é então passada para camadas MLP, que produz as probabilidades finais das classes, como mostrado na imagem.

Enquanto CNNs aprendem hierarquias espaciais de forma implícita por meio de convoluções locais e *pooling*, os ViTs não possuem viés indutivo espacial forte. Isso significa que eles dependem mais fortemente dos dados para aprender relações espaciais, mas, em contrapartida, ganham flexibilidade e capacidade de modelar contextos globais complexos.

Essa diferença explica por que ViTs geralmente exigem grandes volumes de dados e alto poder computacional para treinamento, sendo mais custosos que CNNs tradicionais. Em datasets pequenos, CNNs costumam superar ViTs justamente por incorporarem conhecimento estrutural da imagem desde a arquitetura [<a href="#ref3">3</a>].

Apesar do maior custo e complexidade, o uso de ViT se justifica em diversos cenários, tais como:

|Cenários de Aplicação - ViT|
|--|
|I. Grandes volumes de dados disponíveis para treinamento.|
|II. O problema exigem modelagem global da imagem, indo além de padrões locais.|
|III. Quando os dados de treinamento (tamanho do conjunto e diversidade) e o modelo a ser utilizado (quantidade de parâmetros e complexidade) são grandes.|
|IV. Unificação arquitetural, permitindo o uso de Transformers em múltiplas modalidades (visão, texto, sinais, etc).|

Além disso, variantes modernas como Hybrid CNN-ViT [<a href="#ref4">4</a>], Swin Transformer [<a href="#ref5">5</a>] e ViTs pré-treinados [<a href="#ref6">6</a>] mitigam parte das desvantagens iniciais, tornando essa abordagem cada vez mais prática e competitiva.

> OBSERVAÇÃO: O conjunto de dados utilizado (a) e a arquitetura de ViT implementada (b) não representam os cenários mais adequados para essa aplicabilidade. No entanto, por razões práticas e didáticas, optou-se por utilizar (a) um conjunto de dados reduzido, o que facilita a execução do laboratório em sala de aula, e (b) uma arquitetura simples, construída de forma mais manual, com o objetivo de reforçar a aprendizagem dos conceitos. Essa escolha também se deve às limitações de poder computacional disponíveis, que inviabilizam o uso de arquiteturas ViT mais complexas.

### Metodologia

Neste laboratório, será implementado um modelo ViT feito de forma manual e sem pré-treinamento, baseado no material mostrado no artigo *"An Image is Worth 16x16 Words"* [<a href="#ref7">7</a>]. A ideia central desse material consiste em:

1. **Sem camadas convolucionais**: Sem camadas de convolução para extrair mapas de características.
2. **Patches como palavras**: A imagem será dividida em quadrados ($4 \times 4$ pixels). Para o modelo, cada quadrado é como se fosse uma "palavra" em uma frase.
3. **Atenção Global**: O mecanismo de *self-attention* permite que o modelo relacione o canto superior esquerdo da imagem com o canto inferior direito mais rapidamente, de modo a encontrar sentindo entre os *patches*, algo que uma CNN precisaria de  muitas camadas para fazer.

Desse modo, o objetivo das implementações a seguir é treinar uma ViT no dataset CIFAR-10 e visualizar como ele aprende a geometria e o foco visual.

<figure style="text-align: center;">
  <img src="assets/figura-2.jpg" alt="Figura 1" width="800">
</figure>

**Figura 2** – Rotulações do conjunto de dados CIFAR-10.

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import torchvision
import torch

import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import random

In [ ]:
# Configuração para reprodutibilidade  do experimento
torch.manual_seed(42)
np.random.seed(42)

# Verificando o uso da GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

### Criação de um Tokenizador Visual

Diferente de textos, imagens são grids 2D de pixels. O primeiro desafio é transformar essa matriz $(C, H, W)$ em uma sequência linear de vetores, que é o formato que o Transformer entende. Se temos uma imagem $32 \times 32$ e usamos *patches* de tamanho $4$, teremos uma grade de $8 \times 8$, resultando em 64 *patches*.

Em vez de usar loops `for` para cortar a imagem, será utilizada uma camada `Conv2d` com `kernel_size` e `stride` iguais ao tamanho do patch. Isso realiza a extração e a projeção linear em uma única operação otimizada na GPU.

In [ ]:
class PatchEmbedding(nn.Module):
    # converte imagem 2d em uma sequência de vetores (patches)
    
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=64):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        
        # uso de procedimentos de convolução extrair patches rapidamente
        self.proj = nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x) # (Batch, Embed_Dim, H_patches, W_patches)
        x = x.flatten(2) # (Batch, Embed_Dim, N_Patches)
        x = x.transpose(1, 2) # (Batch, N_Patches, Embed_Dim)
        
        return x

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  No código, a divisão da imagem em patches é feita utilizando uma camada nn.Conv2d com kernel_size e stride iguais ao tamanho do patch. Por que essa abordagem é computacionalmente preferível a usar loops for tradicionais para recortar a imagem? O que aconteceria com a dimensão da sequência de saída se aumentássemos o patch_size de 4 para 8?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
 Se a entrada é uma imagem $32 \times 32$ e o tamanho do patch é 4, temos 64 patches. Se utilizássemos uma imagem de entrada típica do ImageNet ($224 \times 224$) com um patch de $16 \times 16$, qual seria o comprimento da sequência de tokens resultante (sem contar o token CLS)?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

### Modelagem Arquitetural

Existem três componentes críticos que diferenciam o ViT de redes clássicas:

1.  **Token [CLS] (Classification Token):**
    O Transformer processa todos os *patches* em paralelo e devolve uma sequência do mesmo tamanho. Para realizar a classificação final (dizer "é um cachorro"), precisamos de um vetor único que resuma a imagem inteira. Adicionamos um vetor "vazio" (aprendível) no início da fila para servir de "cesto" onde o modelo depositará a informação agregada.

2.  **Positional Embeddings (Geometria):**
    Transformers são invariantes à permutação. Se embaralharmos os *patches*, o cálculo da Atenção é o mesmo. O modelo não sabe o que é "cima" ou "esquerda". Por isso, somamos vetores aprendíveis a cada patch para "ensinar" ao modelo a posição de cada pedaço na imagem original.

3.  **Transformer Encoder:**
    A pilha de camadas que aplica *Multi-Head Attention* e *MLPs*, permitindo que os *patches* troquem informações entre si.

In [ ]:
class VisionTransformer(nn.Module):
    # aplica o uso do modelo ViT em uma tarefa de classificação.
    
    def __init__(self, img_size=32, patch_size=4, num_classes=10, embed_dim=128, depth=6, 
                 num_heads=8, mlp_ratio=4., dropout=0.1):
        super().__init__()
        
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        num_patches = self.patch_embed.n_patches
        
        # tokens e Embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout)

        # encoder Principal
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=int(embed_dim * mlp_ratio), 
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # head de Classificação
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        batch_size = x.shape[0]
        x = self.patch_embed(x)
        
        # adicionar CLS Token e Positional Embeddings
        cls_token = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # pass forward
        x = self.transformer_encoder(x)
        
        # classificar apenas com o token [CLS] (índice 0)
        cls_output = x[:, 0] 
        x = self.norm(cls_output)
        
        return self.head(x)

In [ ]:
def train_model(model, trainloader, testloader, epochs=10, lr=3e-4):
    # treina o modelo ViT utilizando uma base de dados carregada via dataloaders
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    print(f"Iniciando Treino | Params: {sum(p.numel() for p in model.parameters()):,}")
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
        scheduler.step()
        print(f"Epoca {epoch+1}/{epochs} | Loss: {total_loss/len(trainloader):.4f} | Acc: {100.*correct/total:.2f}%")
        
    return model

A função `get_dataloaders()` é responsável por preparar o conjunto de dados CIFAR-10. São realizados três etapas principais:

1. Define as transformações aplicadas às imagens;
2. Carrega os conjuntos de treinamento e teste;
3. Cria os `DataLoader`, que organizam os dados em mini-batches durante o treinamento.

No conjunto de treinamento são utilizadas técnicas de *data augmentation*, como `RandomCrop` e `RandomHorizontalFlip`. Essas transformações geram pequenas variações das imagens originais e ajudam o modelo a generalizar melhor.

Depois, `ToTensor()` converte as imagens para tensores PyTorch e escala os valores dos pixels de `[0, 255]` para `[0, 1]`.

Por fim, `Normalize()` padroniza cada canal de cor utilizando a média e o desvio-padrão calculados para o CIFAR-10. A normalização é dada por:

$$
x' = \frac{x-\mu}{\sigma}
$$

em que $x$ representa o valor do pixel, $\mu$ a média do canal e $\sigma$ o seu desvio-padrão. Assim, os valores dos pixels são aproximadamente centralizados em zero e colocados em uma escala de desvio-padrão unitário.

Os `DataLoader` dividem os dados em mini-batches de tamanho `batch_size`. No treinamento, `shuffle=True` embaralha as amostras a cada época, reduzindo a possibilidade de o modelo aprender padrões relacionados à ordem dos dados. No teste, `shuffle=False` é utilizado, pois a ordem das amostras não precisa ser alterada.

A função retorna os `trainloader` e `testloader`, utilizados no treinamento e avaliação do modelo, além do `testset`, que pode ser utilizado posteriormente para acessar as imagens e seus respectivos rótulos.

In [ ]:
import torchvision.datasets as datasets

cifar_trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())

# item[0] e item[1] são imagens e seus rótulos, respectivamente
imgs = [item[0] for item in cifar_trainset] 
imgs = torch.stack(imgs, dim=0).numpy()

# calcula a média em cada canal(r,g,b)
mean_r = imgs[:,0,:,:].mean()
mean_g = imgs[:,1,:,:].mean()
mean_b = imgs[:,2,:,:].mean()
print(f"Média | R = {mean_r}, G = {mean_g} e B = {mean_b}")

# calcula o desvio padrão em cada canal (r,g,b)
std_r = imgs[:,0,:,:].std()
std_g = imgs[:,1,:,:].std()
std_b = imgs[:,2,:,:].std()
print(f"Desvio Padrão | R = {std_r}, G = {std_g} e B = {std_b}")

In [ ]:
def get_dataloaders(batch_size=128):
    # criação de dataloaders com base nos dados do CIFAR10.
    
    # dados de treinamento com aumento de dados
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((std_b, mean_g, mean_b), (std_r, std_g, std_b)),
    ])

    # dados de teste sem aumento de dados
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    return trainloader, testloader, testset

### Treinamento do Modelo

ViTs são conhecidos por não terem viés indutivo (não assumem prioridades sobre espaço como as CNNs). Isso significa que eles precisam de muitos dados ou muito tempo para aprender que pixels vizinhos são importantes. Neste laboratório, com um dataset pequeno e poucas épocas, espera-se uma acurácia menor que uma ResNet. Entretanto, esse detalhe será irrelevante, pois o objetivo é observar o processo de aprendizado da arquitetura e suas características construtivas.

> OBSERVAÇÃO: O modelo já foi previamente treinado, e os parâmetros obtidos nesse processo serão disponibilizados para carregamento direto no modelo, o que permitirá economizar tempo em sala de aula. No entanto, fique à vontade para descomentar as linhas de código referentes ao treinamento caso deseje realizar modificações, novos experimentos ou outros testes de seu interesse.

In [ ]:
# Carregando os dataloaders
train_loader, test_loader, raw_testset = get_dataloaders()

# Inicialização do modelo
model = VisionTransformer(
    img_size=32, patch_size=4, embed_dim=256, depth=6, num_heads=8, dropout=0.1
).to(DEVICE)

# Treinamento do modelo
#model = train_model(model, train_loader, test_loader, epochs=40)

In [ ]:
# Salvando os pesos do modelo treinado
# torch.save(model.state_dict(), 'vit_cifar10_pretrained.pth')

In [ ]:
# Carregando o modelo treinado
state_dict = torch.load('assets/vit_cifar10_pretrained.pth', map_location=DEVICE)
model.load_state_dict(state_dict, strict=True)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 3:</b><br>
  O texto menciona que as CNNs possuem um viés indutivo espacial forte, enquanto os ViTs não. Com base nisso e nos resultados de acurácia obtidos no treinamento, explique por que o ViT teve dificuldade em generalizar neste dataset pequeno (CIFAR-10) em comparação ao desempenho típico de uma CNN.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>


### Visualização da Decomposição em Patches

Para processar a imagem como uma sequência, o ViT a particiona rigidamente em uma grade não sobreposta. O código abaixo ilustra como o modelo enxerga a entrada inicial: uma coleção de blocos independentes (tokens). Cada quadrado separado pelas linhas brancas será transformado em um vetor e processado em paralelo, cabendo ao mecanismo de Atenção reconstruir o contexto global da imagem a partir desses fragmentos.

In [ ]:
# Carregando o conjunto de dados para visualização
viz_set = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transforms.ToTensor())

In [ ]:
def visualize_patches_grid(dataset, patch_size=4):
    # mostra como uma imagem aleatória é fatiada em patches
    
    idx = random.randint(0, len(dataset) - 1)
    img, label = dataset[idx]

    c, h, w = img.shape
    num_patches_h = h // patch_size
    num_patches_w = w // patch_size
    
    fig, axs = plt.subplots(
        num_patches_h,
        num_patches_w,
        figsize=(4, 4)
    )

    img_np = img.permute(1, 2, 0).numpy()
    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    
    for i in range(num_patches_h):
        for j in range(num_patches_w):
            patch = img_np[
                i * patch_size:(i + 1) * patch_size,
                j * patch_size:(j + 1) * patch_size,
                :
            ]
            axs[i, j].imshow(patch)
            axs[i, j].axis('off')

            # borda branca só para destacar os patches
            for spine in axs[i, j].spines.values():
                spine.set_edgecolor('white')
                spine.set_linewidth(1)
    
    plt.suptitle(
        f"Imagem (idx={idx}) – Patches {patch_size}×{patch_size}",
        fontsize=10
    )
    plt.show()

In [ ]:
visualize_patches_grid(viz_set)

### Análise da Geometria Aprendida

Ao transformar a imagem em uma sequência linear de vetores, perdemos a referência explícita de vizinhança espacial (o modelo não recebe coordenadas X, Y, apenas uma fila de tokens). Teoricamente, o patch 0 e o patch 8 poderiam ser tratados como desconexos, embora na imagem original um esteja logo acima do outro.

A visualização abaixo, organizada como uma grade de mapas de calor, revela que o modelo redescobriu essa estrutura 2D sozinho. Cada pequeno quadrado na grade representa um patch específico, e o mapa de cores dentro dele mostra a similaridade do *embedding* posicional desse patch em relação a todas as outras posições da imagem.

Note a consistência no deslocamento da área amarela (alta similaridade). Para os *patches* do canto superior esquerdo (início da grade), o foco de similaridade também se concentra no topo esquerdo. Conforme percorremos a grade, o brilho acompanha a posição correspondente. Isso prova que o Transformer aprendeu a topologia da imagem: ele deduziu que *patches* vizinhos espacialmente devem ter representações similares, reconstruindo a noção de proximidade 2D sem nunca ter recebido coordenadas explícitas.

In [ ]:
def visualize_position_sim_grid(model, grid_size=8):
    # visualiza a similaridade posicional como uma grade de heatmaps 2d.
    
    # extração e normalização dos embeddings
    pos_embed = model.pos_embed.data[0, 1:, :].cpu() 
    
    # normalização para cosseno (L2 norm)
    pos_embed_norm = pos_embed / pos_embed.norm(dim=1, keepdim=True)
    
    # calcula a Matriz de Similaridade (N_patches x N_patches)
    similarity = pos_embed_norm @ pos_embed_norm.t()
    
    vmin, vmax = similarity.min(), similarity.max()

    fig, axs = plt.subplots(grid_size, grid_size, figsize=(8, 8))
    
    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    

    # itera sobre cada posição da grade
    for i in range(grid_size):
        for j in range(grid_size):
            # conversão de  coordenadas 2D (i, j) para índice linear (0 a 63)
            idx = i * grid_size + j
            
            # separa a linha de similaridade desse patch contra todos os outros
            sim_map = similarity[idx, :]
            
            # remodelagem de volta para 2D (8x8)
            sim_map_2d = sim_map.reshape(grid_size, grid_size).numpy()
            
            ax = axs[i, j]
            ax.imshow(sim_map_2d, cmap="viridis", vmin=vmin, vmax=vmax, origin='upper')
            
            # limpeza visual (remove eixos)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.axis('off')

    plt.suptitle("Similaridade de Embedding Posicional (Topologia Aprendida)", y=0.93, fontsize=14)
    plt.show()

In [ ]:
visualize_position_sim_grid(model, grid_size=8)

Para fins de comparação, a seguir será apresentada a mesma visualização gráfica, desta vez utilizando um modelo não treinado.

In [ ]:
# instanciando uma versão do modelo não treinada
untrained_model = VisionTransformer(
    img_size=32, patch_size=4, embed_dim=256, 
    num_heads=8, dropout=0.1
).to(DEVICE)

# visualizando a similaridade posicional com um modelo não treinado para efeitos comparativos
visualize_position_sim_grid(untrained_model, grid_size=8)

Analisando comparativamente os dois gráficos, podemos tirar duas conclusões sobre as visualizações:

* **Diagonal principal (Auto-Similaridade)**: É matematicamente esperado que o ponto de maior brilho (amarelo intenso) seja sempre o próprio *patch* comparado com ele mesmo, pois a similaridade do cosseno entre dois vetores idênticos é sempre 1.0.
* **Destaque para patches vizinhos**: Note que para a visualização do modelo treinado, os *patches* vizinhos agora possuem cores mais quentes (alta similaridade) do que os *patches* distantes. O modelo aprendeu a relacionar os *embeddings* posicionais próximos com características similares, reconstruindo a noção da continuidade espacial que não existia na inicialização aleatória.

### Análise da Atenção do Modelo

O mapa de calor abaixo mostra quais partes da imagem foram mais relevantes para o modelo realizar as predições das classes. Como o modelo treinado foi feito para ser didático, ele não possui um desempenho muito bom (~63%), de modo que os *patches* considerados como sendo mais relevante vai apresentar um foco difuso e mais grosseiro. Diferente de um modelo SOTA (State-of-the-Art), este ViT ainda distribui parte de sua atenção para o fundo (áreas roxas/alaranjadas), indicando que ele ainda não aprendeu a separar perfeitamente o objeto do cenário. Mesmo com esses problemas, dá para notar que o modelo consegue identificar a presença dos objetos nos cenários.

In [ ]:
def visualize_attention_masked(model, dataset):
    # visualiza o foco do modelo aplicando uma máscara na imagem original
    
    model.eval()
    
    # seleção de uma imagem aleatória
    idx = random.randint(0, len(dataset) - 1)
    img, label = dataset[idx]
    
    x = img.unsqueeze(0).to(DEVICE)
    
    # forward pass manual (extração de features)
    x_patches = model.patch_embed(x)
    cls_token = model.cls_token.expand(x_patches.shape[0], -1, -1)
    x_input = torch.cat((cls_token, x_patches), dim=1) + model.pos_embed
    x_enc = model.transformer_encoder(x_input)
    
    # calcular a similaridade entre CLS e os patches
    cls_vec = x_enc[0, 0, :] # Vetor final de classificação
    patch_vecs = x_enc[0, 1:, :] # Vetores finais dos patches
    
    # produto escalar para ver quais patches alinham com a decisão final
    attn_score = torch.matmul(patch_vecs, cls_vec)
    
    # reshape para grade 2D
    attn_grid = attn_score.reshape(8, 8)
    
    # normalização Min-Max para garantir intervalo [0, 1]
    attn_grid = (attn_grid - attn_grid.min()) / (attn_grid.max() - attn_grid.min())
    
    # transforma o 8x8 em 32x32 usando interpolação bicúbica para ficar suave
    attn_mask = F.interpolate(
        attn_grid.unsqueeze(0).unsqueeze(0), 
        size=(32, 32), 
        mode='bicubic', 
        align_corners=False
    ).squeeze().detach().cpu().numpy()

    # a interpolação bicúbica pode gerar valores < 0 ou > 1. Forçamos o limite
    attn_mask = np.clip(attn_mask, 0, 1)
    
    # aplicar a Máscara na imagem original
    img_np = img.permute(1, 2, 0).cpu().numpy()
    
    # desnormalizar a imagem original para visualização correta
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    # multiplicação: Imagem * Máscara (broadcasting)
    mask_rgb = np.stack([attn_mask]*3, axis=-1)
    
    alpha = 0.95 
    masked_img = (1 - alpha) * img_np + alpha * (img_np * mask_rgb)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
    
    ax1.imshow(img_np)
    ax1.set_title(f"Original: {dataset.classes[label]}")
    ax1.axis('off')
    
    # plot do Heatmap puro
    sns.heatmap(attn_grid.detach().cpu().numpy(), cmap='inferno', square=True, ax=ax2, cbar=False)
    ax2.set_title("Heatmap (Patches 8x8)")
    ax2.axis('off')
    
    ax3.imshow(masked_img)
    ax3.set_title("Máscara de Atenção")
    ax3.axis('off')

    plt.show()

visualize_attention_masked(model, viz_set)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 4:</b><br>
  O mecanismo de Self-Attention escala quadraticamente com o número de tokens ($O(N^2)$). Considerando isso, discuta a viabilidade de usar ViTs puros para imagens de altíssima resolução (ex: imagens médicas de $4000 \times 4000$ pixels) versus uma abordagem híbrida (CNN + ViT). Pesquise um pouco sobre algumas soluções híbridas que mesclam a extração de características usando camadas convolucionais e recursos de ViT.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

### Porque ViTs foram revolucionárias?

Embora o ViT possa apresentar desempenho inferior a uma CNN em modelos pequenos e com conjuntos de dados limitados, sua principal contribuição está na capacidade de escalar com grandes volumes de dados e recursos computacionais.

As CNNs possuem viéses indutivos específicos para imagens, como localidade e compartilhamento de pesos. Esses vieses são vantajosos quando há poucos dados, pois incorporam ao modelo algumas propriedades que sabemos que são comuns em imagens. O ViT, por outro lado, possui um viés indutivo mais fraco e, portanto, precisa aprender mais dessas relações diretamente a partir dos dados.

Essa característica pode ser vista como uma desvantagem em problemas pequenos, mas torna-se uma vantagem quando a quantidade de dados aumenta. Com dados suficientes e uma construção arquitetural igualmente adequada, o ViT consegue aprender representaçõs visuais complexas sem depender tanto de restrições impostas pela modelagem arquitetural [<a href="#ref1">1</a>, <a href="#ref2">2</a>].

Esse comportamento foi fundamental para a evolução dos atuais modelos de visão de grande escala e, posteriormente, de modelos multimodais que combinam representações visuais e textuais.

## Referências

<p id="ref1">[1] Raghu, M., Unterthiner, T., Kornblith, S., Zhang, C., & Dosovitskiy, A. (2021). Do vision transformers see like convolutional neural networks?. Advances in neural information processing systems, 34, 12116-12128.
<p id="ref2">[2] Han, K., Wang, Y., Chen, H., Chen, X., Guo, J., Liu, Z., ... & Tao, D. (2022). A survey on vision transformer. IEEE transactions on pattern analysis and machine intelligence, 45(1), 87-110.
<p id="ref3">[3] Steiner, A., Kolesnikov, A., Zhai, X., Wightman, R., Uszkoreit, J., & Beyer, L. (2021). How to train your vit? data, augmentation, and regularization in vision transformers. arXiv preprint arXiv:2106.10270.
<p id="ref4">[4] Pantelaios, D., Theofilou, P. A., Tzouveli, P., & Kollias, S. (2024, May). Hybrid CNN-ViT models for medical image classification. In 2024 IEEE international symposium on biomedical imaging (ISBI) (pp. 1-4). IEEE.
<p id="ref5">[5] Liu, Z., Lin, Y., Cao, Y., Hu, H., Wei, Y., Zhang, Z., ... & Guo, B. (2021). Swin transformer: Hierarchical vision transformer using shifted windows. In Proceedings of the IEEE/CVF international conference on computer vision (pp. 10012-10022).
<p id="ref6">[6] Bao, H., Dong, L., Piao, S., & Wei, F. (2021). Beit: Bert pre-training of image transformers. arXiv preprint arXiv:2106.08254.
<p id="ref7">[7] Dosovitskiy, A. (2020). An image is worth 16x16 words: Transformers for image recognition at scale. arXiv preprint arXiv:2010.11929.